# Data Generation

## Imports

In [1]:
import os
import numpy as np
import pandas as pd
from scipy.io import loadmat

## Directories

In [2]:
base_dir = os.getcwd()
data_dir = os.path.join(base_dir, 'data')

processed_data_dir = os.path.join(base_dir, 'processed_data')
os.makedirs(processed_data_dir, exist_ok=True)

## Data generation settings

In [10]:
T_limit = 1000
train_val_test_split = [0.7, 0.15, 0.15]
window_length, stride = 30, 1
num_chunks = 90

## Load Data

In [4]:
ampds_folderpath = os.path.join(data_dir, 'ampds')
ampds_filepath = os.path.join(ampds_folderpath, 'ampds2.mat')

data = loadmat(ampds_filepath)

x, y = data['input'], data['output']
input_labels, output_labels = data['labelInp'], data['labelOut']

T = x.shape[0]
T_limit = min(T, T_limit) if T_limit is not None else T
x, y = x[:T_limit], y[:T_limit]

x_df = pd.DataFrame(x, columns=input_labels)
y_P_df = pd.DataFrame(y[:,:,0], columns=output_labels)

print("Input shape:", x_df.shape)
print("Output shape:", y_P_df.shape)
print("Input missing values:")
display(x_df.isna().sum())
print("Output missing values:")
display(y_P_df.isna().sum())

display(x_df.head())
display(y_P_df.head())

Input shape: (1000, 6)
Output shape: (1000, 22)
Input missing values:


time     0
id       0
P-agg    0
I-agg    0
Q-agg    0
S-agg    0
dtype: int64

Output missing values:


time    0
id      0
RSE     0
GRE     0
B1E     0
BME     0
CWE     0
DWE     0
EQE     0
FRE     0
HPE     0
OFE     0
UTE     0
WOE     0
B2E     0
CDE     0
DNE     0
EBE     0
FGE     0
HTE     0
OUE     0
TVE     0
dtype: int64

,time,id,P-agg,I-agg,Q-agg,S-agg
0,60.0,1.0,918.0,8.3,141.0,1037.0
1,120.0,1.0,913.0,8.2,141.0,1028.0
2,180.0,1.0,872.0,8.0,139.0,993.0
3,240.0,1.0,872.0,8.0,141.0,990.0
4,300.0,1.0,772.0,7.8,140.0,988.0


,time,id,RSE,GRE,B1E,BME,CWE,DWE,EQE,FRE,...,UTE,WOE,B2E,CDE,DNE,EBE,FGE,HTE,OUE,TVE
0,60.0,1.0,43.0,2.0,0.0,352.0,0.0,0.0,41.0,106.0,...,9.0,0.0,58.0,0.0,30.0,0.0,0.0,4.0,0.0,37.0
1,120.0,1.0,43.0,0.0,0.0,354.0,0.0,0.0,35.0,107.0,...,9.0,0.0,58.0,0.0,30.0,0.0,0.0,4.0,0.0,37.0
2,180.0,1.0,43.0,2.0,0.0,354.0,0.0,0.0,41.0,107.0,...,9.0,0.0,8.0,0.0,30.0,0.0,0.0,4.0,0.0,37.0
3,240.0,1.0,43.0,2.0,0.0,354.0,0.0,0.0,40.0,106.0,...,9.0,0.0,7.0,0.0,30.0,0.0,0.0,4.0,0.0,37.0
4,300.0,1.0,41.0,2.0,0.0,352.0,0.0,0.0,40.0,106.0,...,9.0,0.0,8.0,0.0,31.0,0.0,0.0,4.0,0.0,37.0


## Normalize data

Min–max normalization rescales each feature using its own minimum and maximum:

$
x_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}
$

This maps values to roughly 0–1. The `1e-8` term prevents division by zero when a feature is constant.

It’s often used before machine-learning training so features with larger numeric ranges don’t dominate, and to help some models train more reliably. Calculate min/max on the training data, then reuse those same values for validation and test data.

In [5]:
def normalize_data(x):
    x_min = x.min(axis=0, keepdims=True)
    x_max = x.max(axis=0, keepdims=True)
    return x_min, x_max, (x - x_min) / (x_max - x_min + 1e-8)

In [6]:
x_min, x_max, x_norm = normalize_data(x)
y_min, y_max, y_norm = normalize_data(y)

scaling_factors = {
    'x_min': x_min,
    'x_max': x_max,
    'y_min': y_min,
    'y_max': y_max}

## Create NILM Dataset

### Precompute indices

This function creates train, validation, and test examples from the time-series data without letting and timestamp belong simultaneously to multiple splits.

1. Divide the timeline into chunks: `boundaries` marks the edges of `num_chunks` contiguous chunks. If the timeline lenfth doesn't divide evenly by the number of chunks, some chunks get one extra timestep.

2. Randomly assign chunks to splits: `chunk_counts` allocates whole chunks to the train/validation/test splits, as close as possible to the requested proportions. 

3. Make windows inside each assigned chunk: Each `inp_idx` is a window's start position. Its matching `out_idx` is the window's center position. Only windows fully inside a chunk are included. This prevents chunks from simultaneously belonging to multiple splits.

4. Shuffle exmaples within each split.

5. Return the results: `idx_dict` maps each split name to `(input_indices, output_indices)`. `num_windows` is the total number of valid examples across all splits.

In [12]:
def precompute_indices(
    num_timesteps,
    window_length,
    stride,
    train_val_test_split,
    num_chunks,
    seed=42):

    # Divide the timeline into contiguous, non-overlapping chunks.
    boundaries = np.linspace(0, num_timesteps, num_chunks + 1, dtype=int)
    chunk_ids = np.random.default_rng(seed).permutation(num_chunks)

    # Allocate whole chunks to splits, as close as possible to the requested ratios.
    exact_counts = np.asarray(train_val_test_split) * num_chunks
    chunk_counts = np.floor(exact_counts).astype(int)
    remainder = num_chunks - chunk_counts.sum()
    chunk_counts[np.argsort(-(exact_counts - chunk_counts))[:remainder]] += 1

    split_names = ("train", "val", "test")
    idx_dict = {}
    chunk_offset = 0

    for split_name, chunk_count in zip(split_names, chunk_counts):
        split_chunk_ids = chunk_ids[chunk_offset:chunk_offset + chunk_count]
        chunk_offset += chunk_count

        inp_parts = []
        out_parts = []

        for chunk_id in split_chunk_ids:
            chunk_start = boundaries[chunk_id]
            chunk_end = boundaries[chunk_id + 1]

            # Keep only windows fully contained in this chunk.
            starts = np.arange(
                chunk_start,
                chunk_end - window_length + 1,
                stride,
            )
            inp_parts.append(starts)
            out_parts.append(starts + window_length // 2)

        inp_idx = np.concatenate(inp_parts) if inp_parts else np.array([], dtype=int)
        out_idx = np.concatenate(out_parts) if out_parts else np.array([], dtype=int)

        # Shuffle examples within each split.
        perm = np.random.default_rng(seed + len(idx_dict)).permutation(len(inp_idx))
        idx_dict[split_name] = (inp_idx[perm], out_idx[perm])

    num_windows = sum(len(indices[0]) for indices in idx_dict.values())
    return num_windows, idx_dict


num_windows, idx_dict = precompute_indices(
    len(x), window_length, stride, train_val_test_split, num_chunks, seed=42)

In [15]:
display(pd.DataFrame([
    {"Split": split, "Windows": len(inputs)}
    for split, (inputs, outputs) in idx_dict.items()]))

,Split,Windows
0,train,497
1,val,142
2,test,71


### Save dataset

`ampds_dataset.npz` is a compressed NumPy archive containing:
-  `x_norm` and `y_norm`: normalized input and target data.
- `scaling factors`: The min/max values used to normalize `x` and `y`, for reversing the normalization later. 
- `train_inp / train_out`, `val_inp` / `val_out`, `test_inp / test_out`: index arrays assigning each frame to a split, and designating the corresponding target.
- `T_limit`, `train_val_test_split`, `window_length`, `stride`: parameters describing the data limit, split proportions, and windowing setup.

In [16]:
dataset_path = os.path.join(processed_data_dir, 'ampds_dataset.npz')

np.savez_compressed(
    dataset_path,
    x_norm=x_norm,
    y_norm=y_norm,
    **scaling_factors,
    train_inp=idx_dict["train"][0], train_out=idx_dict["train"][1],
    val_inp=idx_dict["val"][0], val_out=idx_dict["val"][1],
    test_inp=idx_dict["test"][0], test_out=idx_dict["test"][1],
    T_limit=T_limit,
    train_val_test_split=train_val_test_split,
    window_length=window_length,
    stride=stride)